# Reto 8: Analizador de Precios de Acciones

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional
### Febrero - Julio 2026

---

## Contexto del Problema

Una casa de bolsa necesita un sistema para **analizar el comportamiento histórico de precios de acciones** usando Pandas Series. El sistema debe calcular métricas de rendimiento, identificar tendencias y generar alertas.

Tu tarea es crear un **analizador de acciones** que procese datos de precios y genere insights útiles para inversionistas.

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    ANALIZADOR DE ACCIONES                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ENTRADA                                                               │
│   ───────                                                               │
│   Series de precios de cierre diarios                                   │
│   • Índice: fechas                                                      │
│   • Valores: precios de cierre ($)                                      │
│                                                                         │
│   ANÁLISIS                                                              │
│   ────────                                                              │
│   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐                  │
│   │Estadísticas │ → │ Rendimiento │ → │  Señales    │                  │
│   │  básicas    │   │  y riesgo   │   │  trading    │                  │
│   └─────────────┘   └─────────────┘   └─────────────┘                  │
│                                                                         │
│   SALIDA                                                                │
│   ──────                                                                │
│   • Reporte de rendimiento                                              │
│   • Indicadores técnicos                                                │
│   • Alertas de compra/venta                                             │
│   • Clasificación de volatilidad                                        │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

## Conceptos Financieros Básicos

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    CONCEPTOS CLAVE                                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   RENDIMIENTO DIARIO                                                    │
│   ───────────────────                                                   │
│   rendimiento = (precio_hoy - precio_ayer) / precio_ayer × 100          │
│                                                                         │
│   MEDIA MÓVIL (Moving Average)                                          │
│   ────────────────────────────                                          │
│   Promedio de los últimos N días                                        │
│   MA_5 = promedio de últimos 5 precios                                  │
│                                                                         │
│   VOLATILIDAD                                                           │
│   ───────────                                                           │
│   Desviación estándar de los rendimientos                               │
│   Mayor volatilidad = mayor riesgo                                      │
│                                                                         │
│   SEÑALES DE TRADING (simplificado)                                     │
│   ─────────────────────────────────                                     │
│   • COMPRA: Precio cruza MA hacia arriba                                │
│   • VENTA: Precio cruza MA hacia abajo                                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

## Requerimientos Funcionales

### Parte 1: Análisis Estadístico Básico (30%)

```python
def estadisticas_basicas(precios: pd.Series) -> dict:
    """
    Calcula estadísticas descriptivas de los precios.
    """
    resultado = {
        "precio_actual": precios.iloc[-1],
        "precio_minimo": precios.min(),
        "precio_maximo": precios.max(),
        "precio_promedio": precios.mean(),
        "precio_mediana": precios.median(),
        "desviacion_std": precios.std(),
        "rango": precios.max() - precios.min(),
        "dias_analizados": len(precios)
    }
    return resultado

def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.
    """
    return precios.pct_change() * 100

def analisis_rendimientos(rendimientos: pd.Series) -> dict:
    """
    Analiza los rendimientos calculados.
    """
    rend_limpios = rendimientos.dropna()
    mejor_idx = rend_limpios.idxmax()
    peor_idx = rend_limpios.idxmin()
    
    fecha_mejor = mejor_idx.strftime('%Y-%m-%d') if hasattr(mejor_idx, 'strftime') else str(mejor_idx)[:10]
    fecha_peor = peor_idx.strftime('%Y-%m-%d') if hasattr(peor_idx, 'strftime') else str(peor_idx)[:10]
    
    resultado = {
        "rendimiento_total": rend_limpios.sum(),
        "rendimiento_promedio": rend_limpios.mean(),
        "mejor_dia": (fecha_mejor, rend_limpios.max()),
        "peor_dia": (fecha_peor, rend_limpios.min()),
        "dias_positivos": (rend_limpios > 0).sum(),
        "dias_negativos": (rend_limpios < 0).sum(),
        "volatilidad": rend_limpios.std()
    }
    return resultado

### Parte 2: Indicadores Técnicos (35%)

```python
def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la media móvil simple (SMA).
    """
    return precios.rolling(window=ventana).mean()

def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> dict:
    """
    Calcula las Bandas de Bollinger.
    """
    sma = precios.rolling(window=ventana).mean()
    std_dev = precios.rolling(window=ventana).std()
    
    return {
        "banda_superior": sma + (num_std * std_dev),
        "banda_media": sma,
        "banda_inferior": sma - (num_std * std_dev)
    }

def detectar_maximos_minimos(precios: pd.Series, ventana: int = 5) -> dict:
    """
    Detecta máximos y mínimos locales.
    """
    roll_max = precios.rolling(window=2*ventana+1, center=True, min_periods=1).max()
    roll_min = precios.rolling(window=2*ventana+1, center=True, min_periods=1).min()
    
    return {
        "maximos": precios[precios == roll_max],
        "minimos": precios[precios == roll_min]
    }

def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """
    Clasifica la tendencia actual.
    """
    ma = media_movil(precios, ventana)
    
    if ma.isna().all() or len(ma) < 2:
        return "LATERAL"
        
    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2]
    
    if precio_actual > ma_actual and ma_actual > ma_anterior:
        return "ALCISTA"
    elif precio_actual < ma_actual and ma_actual < ma_anterior:
        return "BAJISTA"
    else:
        return "LATERAL"


### Parte 3: Sistema de Alertas (35%)

```python
def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """
    Genera señales de compra/venta basadas en cruces de medias móviles.
    """
    mc = media_movil(precios, ma_corta)
    ml = media_movil(precios, ma_larga)
    
    senales = pd.Series("MANTENER", index=precios.index)
    
    cruce_alcista = (mc > ml) & (mc.shift(1) <= ml.shift(1))
    cruce_bajista = (mc < ml) & (mc.shift(1) >= ml.shift(1))
    
    senales[cruce_alcista] = "COMPRA"
    senales[cruce_bajista] = "VENTA"
    
    return senales

def alertas_precio(precios: pd.Series, umbral_cambio: float = 5.0) -> list[Dict]:
    """
    Genera alertas cuando hay cambios significativos.
    """
    rendimientos = calcular_rendimientos(precios)
    alertas_filtradas = rendimientos[rendimientos.abs() > umbral_cambio].dropna()
    
    alertas = []
    for fecha, cambio in alertas_filtradas.items():
        fecha_str = fecha.strftime('%Y-%m-%d') if hasattr(fecha, 'strftime') else str(fecha)[:10]
        alertas.append({
            "fecha": fecha_str,
            "tipo": "SUBIDA" if cambio > 0 else "CAIDA",
            "cambio": float(cambio)
        })
        
    return alertas

def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """
    Clasifica el nivel de volatilidad del activo.
    """
    std = rendimientos.std()
    
    if pd.isna(std): return "N/A"
    if std < 1.0: return "BAJA"
    elif std <= 3.0: return "MEDIA"
    elif std <= 5.0: return "ALTA"
    else: return "MUY ALTA"

def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> dict:
    """
    Genera un reporte completo de análisis.
    """
    rendimientos = calcular_rendimientos(precios)
    
    fecha_inicio = precios.index[0].strftime('%Y-%m-%d') if hasattr(precios.index[0], 'strftime') else str(precios.index[0])[:10]
    fecha_fin = precios.index[-1].strftime('%Y-%m-%d') if hasattr(precios.index[-1], 'strftime') else str(precios.index[-1])[:10]
    
    reporte = {
        "nombre": nombre_accion,
        "periodo": {
            "inicio": fecha_inicio,
            "fin": fecha_fin,
            "dias": len(precios)
        },
        "estadisticas": estadisticas_basicas(precios),
        "rendimientos": analisis_rendimientos(rendimientos),
        "tendencia": clasificar_tendencia(precios),
        "volatilidad": clasificar_volatilidad(rendimientos),
        "senal_actual": generar_senales_trading(precios).iloc[-1],
        "alertas_recientes": alertas_precio(precios)
    }
    
    return reporte

## Código Base

In [23]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional

In [24]:
# PARTE 1: Análisis Estadístico Básico

def estadisticas_basicas(precios: pd.Series) -> Dict:
    """
    Calcula estadísticas descriptivas de los precios.
    """
    # Se usan métodos propios de pandas Series para obtener cada métrica.
    resultado = {
        "precio_actual": precios.iloc[-1],
        "precio_minimo": precios.min(),
        "precio_maximo": precios.max(),
        "precio_promedio": precios.mean(),
        "precio_mediana": precios.median(),
        "desviacion_std": precios.std(),
        "rango": precios.max() - precios.min(),
        "dias_analizados": len(precios)
    }

    return resultado

In [25]:
def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.
    """
    # pct_change() calcula el cambio relativo entre un precio y el anterior.
    # Se multiplica por 100 para expresarlo como porcentaje.
    return precios.pct_change() * 100

In [26]:
def analisis_rendimientos(rendimientos: pd.Series) -> Dict:
    """
    Analiza los rendimientos calculados.
    """
    # El primer rendimiento queda como NaN porque no hay día anterior.
    # Por eso se elimina antes de calcular máximos, mínimos y promedios.
    rend_limpios = rendimientos.dropna()

    mejor_idx = rend_limpios.idxmax()
    peor_idx = rend_limpios.idxmin()

    fecha_mejor = mejor_idx.strftime('%Y-%m-%d') if hasattr(mejor_idx, 'strftime') else str(mejor_idx)[:10]
    fecha_peor = peor_idx.strftime('%Y-%m-%d') if hasattr(peor_idx, 'strftime') else str(peor_idx)[:10]

    resultado = {
        "rendimiento_total": rend_limpios.sum(),
        "rendimiento_promedio": rend_limpios.mean(),
        "mejor_dia": (fecha_mejor, rend_limpios.max()),
        "peor_dia": (fecha_peor, rend_limpios.min()),
        "dias_positivos": (rend_limpios > 0).sum(),
        "dias_negativos": (rend_limpios < 0).sum(),
        "volatilidad": rend_limpios.std()
    }

    return resultado

In [27]:
# PARTE 2: Indicadores Técnicos

def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la media móvil simple (SMA).
    """
    # rolling(window=ventana) toma bloques de datos consecutivos.
    # mean() calcula el promedio móvil de cada bloque.
    return precios.rolling(window=ventana).mean()

In [28]:
def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> Dict:
    """
    Calcula las Bandas de Bollinger.
    """
    # La banda media es la media móvil.
    sma = precios.rolling(window=ventana).mean()

    # La desviación estándar mide qué tanto se aleja el precio de su media.
    std_dev = precios.rolling(window=ventana).std()

    resultado = {
        "banda_superior": sma + (num_std * std_dev),
        "banda_media": sma,
        "banda_inferior": sma - (num_std * std_dev)
    }

    return resultado

In [29]:
def detectar_maximos_minimos(precios: pd.Series, ventana: int = 5) -> Dict:
    """
    Detecta máximos y mínimos locales.
    """
    # Se usa una ventana centrada para comparar cada precio contra sus vecinos.
    roll_max = precios.rolling(window=2*ventana+1, center=True, min_periods=1).max()
    roll_min = precios.rolling(window=2*ventana+1, center=True, min_periods=1).min()

    resultado = {
        "maximos": precios[precios == roll_max],
        "minimos": precios[precios == roll_min]
    }

    return resultado

In [30]:
def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """
    Clasifica la tendencia actual.
    """
    # Primero se calcula la media móvil para suavizar los precios.
    ma = media_movil(precios, ventana)

    # Si no hay suficientes datos para calcular la media móvil, se considera lateral.
    if ma.isna().all() or len(ma) < 2:
        return "LATERAL"

    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2]

    # Tendencia alcista: precio por encima de la media y media subiendo.
    if precio_actual > ma_actual and ma_actual > ma_anterior:
        return "ALCISTA"

    # Tendencia bajista: precio por debajo de la media y media bajando.
    elif precio_actual < ma_actual and ma_actual < ma_anterior:
        return "BAJISTA"

    # Cualquier otro caso se clasifica como lateral.
    else:
        return "LATERAL"

In [31]:
# PARTE 3: Sistema de Alertas

def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """
    Genera señales de compra/venta basadas en cruces de medias móviles.
    """
    # Calcula la media móvil corta y la media móvil larga.
    mc = media_movil(precios, ma_corta)
    ml = media_movil(precios, ma_larga)

    # Por defecto, todos los días son "MANTENER".
    senales = pd.Series("MANTENER", index=precios.index)

    # Compra: la media corta cruza de abajo hacia arriba la media larga.
    cruce_alcista = (mc > ml) & (mc.shift(1) <= ml.shift(1))

    # Venta: la media corta cruza de arriba hacia abajo la media larga.
    cruce_bajista = (mc < ml) & (mc.shift(1) >= ml.shift(1))

    senales[cruce_alcista] = "COMPRA"
    senales[cruce_bajista] = "VENTA"

    return senales

In [32]:
def alertas_precio(precios: pd.Series, umbral_cambio: float = 5.0) -> List[Dict]:
    """
    Genera alertas cuando hay cambios significativos.
    """
    # Primero se calculan los rendimientos diarios.
    rendimientos = calcular_rendimientos(precios)

    # Se filtran solamente los cambios cuyo valor absoluto supera el umbral.
    alertas_filtradas = rendimientos[rendimientos.abs() > umbral_cambio].dropna()

    alertas = []

    # Cada alerta se guarda como diccionario con fecha, tipo y cambio porcentual.
    for fecha, cambio in alertas_filtradas.items():
        fecha_str = fecha.strftime('%Y-%m-%d') if hasattr(fecha, 'strftime') else str(fecha)[:10]
        alertas.append({
            "fecha": fecha_str,
            "tipo": "SUBIDA" if cambio > 0 else "CAIDA",
            "cambio": float(cambio)
        })

    return alertas

In [33]:
def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """
    Clasifica el nivel de volatilidad del activo.
    """
    # La volatilidad se mide usando la desviación estándar de los rendimientos.
    std = rendimientos.std()

    if pd.isna(std):
        return "N/A"
    if std < 1.0:
        return "BAJA"
    elif std <= 3.0:
        return "MEDIA"
    elif std <= 5.0:
        return "ALTA"
    else:
        return "MUY ALTA"

In [34]:
def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> Dict:
    """
    Genera un reporte completo de análisis.
    """
    # Se calculan los rendimientos una sola vez para reutilizarlos.
    rendimientos = calcular_rendimientos(precios)

    fecha_inicio = precios.index[0].strftime('%Y-%m-%d') if hasattr(precios.index[0], 'strftime') else str(precios.index[0])[:10]
    fecha_fin = precios.index[-1].strftime('%Y-%m-%d') if hasattr(precios.index[-1], 'strftime') else str(precios.index[-1])[:10]

    # El reporte integra estadísticas, rendimientos, tendencia, volatilidad,
    # señal actual y alertas recientes.
    reporte = {
        "nombre": nombre_accion,
        "periodo": {
            "inicio": fecha_inicio,
            "fin": fecha_fin,
            "dias": len(precios)
        },
        "estadisticas": estadisticas_basicas(precios),
        "rendimientos": analisis_rendimientos(rendimientos),
        "tendencia": clasificar_tendencia(precios),
        "volatilidad": clasificar_volatilidad(rendimientos),
        "senal_actual": generar_senales_trading(precios).iloc[-1],
        "alertas_recientes": alertas_precio(precios)
    }

    return reporte

## Datos de Prueba

In [35]:
# Simular 60 días de precios de una acción
np.random.seed(42)  # Para reproducibilidad

# Generar fechas
fechas = pd.date_range(start='2024-01-01', periods=60, freq='B')  # B = días hábiles

# Generar precios con tendencia alcista y volatilidad
precio_inicial = 100
rendimientos_simulados = np.random.normal(0.002, 0.02, 60)  # Media 0.2%, std 2%
precios_simulados = precio_inicial * np.cumprod(1 + rendimientos_simulados)

# Crear Serie de precios
PRECIOS_ACCION = pd.Series(
    precios_simulados.round(2),
    index=fechas,
    name='ACME Corp'
)

print("Precios de ACME Corp (primeros 10 días):")
print(PRECIOS_ACCION.head(10))
print(f"\nTotal de días: {len(PRECIOS_ACCION)}")

Precios de ACME Corp (primeros 10 días):
2024-01-01    101.19
2024-01-02    101.12
2024-01-03    102.63
2024-01-04    105.96
2024-01-05    105.68
2024-01-08    105.39
2024-01-09    108.93
2024-01-10    110.82
2024-01-11    110.00
2024-01-12    111.42
Freq: B, Name: ACME Corp, dtype: float64

Total de días: 60


In [36]:
# Datos adicionales para pruebas más completas
np.random.seed(123)

# Acción con alta volatilidad
rend_volatil = np.random.normal(0, 0.05, 60)  # 5% de volatilidad diaria
precios_volatil = 50 * np.cumprod(1 + rend_volatil)
ACCION_VOLATIL = pd.Series(
    precios_volatil.round(2),
    index=fechas,
    name='VolatilTech'
)

# Acción con tendencia bajista
rend_bajista = np.random.normal(-0.005, 0.015, 60)  # Tendencia negativa
precios_bajista = 200 * np.cumprod(1 + rend_bajista)
ACCION_BAJISTA = pd.Series(
    precios_bajista.round(2),
    index=fechas,
    name='DeclineCorp'
)

print("Acciones disponibles para análisis:")
print(f"1. ACME Corp - Precio actual: ${PRECIOS_ACCION.iloc[-1]:.2f}")
print(f"2. VolatilTech - Precio actual: ${ACCION_VOLATIL.iloc[-1]:.2f}")
print(f"3. DeclineCorp - Precio actual: ${ACCION_BAJISTA.iloc[-1]:.2f}")

Acciones disponibles para análisis:
1. ACME Corp - Precio actual: $92.74
2. VolatilTech - Precio actual: $59.42
3. DeclineCorp - Precio actual: $138.97


## Funciones de Visualización

In [37]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de forma legible."""
    print("=" * 70)
    print(f"           REPORTE DE ANÁLISIS: {reporte['nombre']}")
    print("=" * 70)
    
    # Período
    periodo = reporte.get('periodo', {})
    print(f"\n📅 PERÍODO DE ANÁLISIS")
    print("-" * 40)
    print(f"Inicio: {periodo.get('inicio', 'N/A')}")
    print(f"Fin: {periodo.get('fin', 'N/A')}")
    print(f"Días analizados: {periodo.get('dias', 'N/A')}")
    
    # Estadísticas
    stats = reporte.get('estadisticas', {})
    print(f"\n📊 ESTADÍSTICAS DE PRECIO")
    print("-" * 40)
    print(f"Precio actual:  ${stats.get('precio_actual', 0):,.2f}")
    print(f"Precio mínimo:  ${stats.get('precio_minimo', 0):,.2f}")
    print(f"Precio máximo:  ${stats.get('precio_maximo', 0):,.2f}")
    print(f"Precio promedio: ${stats.get('precio_promedio', 0):,.2f}")
    
    # Rendimientos
    rend = reporte.get('rendimientos', {})
    print(f"\n📈 RENDIMIENTO")
    print("-" * 40)
    print(f"Rendimiento total: {rend.get('rendimiento_total', 0):+.2f}%")
    print(f"Rendimiento promedio diario: {rend.get('rendimiento_promedio', 0):+.3f}%")
    if rend.get('mejor_dia'):
        print(f"Mejor día: {rend['mejor_dia'][0]} ({rend['mejor_dia'][1]:+.2f}%)")
    if rend.get('peor_dia'):
        print(f"Peor día: {rend['peor_dia'][0]} ({rend['peor_dia'][1]:+.2f}%)")
    print(f"Días positivos: {rend.get('dias_positivos', 0)}")
    print(f"Días negativos: {rend.get('dias_negativos', 0)}")
    
    # Indicadores
    print(f"\n🎯 INDICADORES")
    print("-" * 40)
    print(f"Tendencia: {reporte.get('tendencia', 'N/A')}")
    print(f"Volatilidad: {reporte.get('volatilidad', 'N/A')}")
    print(f"Señal actual: {reporte.get('senal_actual', 'N/A')}")
    
    # Alertas
    alertas = reporte.get('alertas_recientes', [])
    if alertas:
        print(f"\n⚠️ ALERTAS RECIENTES")
        print("-" * 40)
        for alerta in alertas[-5:]:  # Últimas 5
            emoji = "🔺" if alerta['tipo'] == 'SUBIDA' else "🔻"
            print(f"{emoji} {alerta['fecha']}: {alerta['tipo']} de {alerta['cambio']:+.2f}%")
    
    print("\n" + "=" * 70)

In [38]:
def visualizar_precios_texto(precios: pd.Series, ancho: int = 50) -> None:
    """Visualización simple de precios en texto (ASCII chart)."""
    min_precio = precios.min()
    max_precio = precios.max()
    rango = max_precio - min_precio
    
    print(f"\nGráfico de precios: {precios.name}")
    print(f"Max: ${max_precio:.2f}")
    print("-" * (ancho + 10))
    
    # Mostrar cada 3 días para no saturar
    for fecha, precio in precios.iloc[::3].items():
        posicion = int((precio - min_precio) / rango * ancho) if rango > 0 else ancho // 2
        barra = " " * posicion + "█"
        fecha_str = fecha.strftime('%m/%d') if hasattr(fecha, 'strftime') else str(fecha)[:5]
        print(f"{fecha_str} |{barra}")
    
    print("-" * (ancho + 10))
    print(f"Min: ${min_precio:.2f}")

## Prueba tu Implementación

In [39]:
# Prueba de funciones individuales
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

# Estadísticas básicas
print("\n-- Estadísticas Básicas --")
stats = estadisticas_basicas(PRECIOS_ACCION)
print(stats)

# Rendimientos
print("\n-- Rendimientos (primeros 5) --")
rendimientos = calcular_rendimientos(PRECIOS_ACCION)
print(rendimientos.head())

# Análisis de rendimientos
print("\n-- Análisis de Rendimientos --")
analisis = analisis_rendimientos(rendimientos)
print(analisis)

PRUEBA DE FUNCIONES INDIVIDUALES

-- Estadísticas Básicas --
{'precio_actual': np.float64(92.74), 'precio_minimo': 86.67, 'precio_maximo': 111.42, 'precio_promedio': np.float64(96.88983333333334), 'precio_mediana': 95.645, 'desviacion_std': 7.128231646939468, 'rango': 24.75, 'dias_analizados': 60}

-- Rendimientos (primeros 5) --
2024-01-01         NaN
2024-01-02   -0.069177
2024-01-03    1.493275
2024-01-04    3.244665
2024-01-05   -0.264251
Freq: B, Name: ACME Corp, dtype: float64

-- Análisis de Rendimientos --
{'rendimiento_total': np.float64(-7.748229086684423), 'rendimiento_promedio': np.float64(-0.1313259167234648), 'mejor_dia': ('2024-02-13', 3.905831995719633), 'peor_dia': ('2024-02-21', -3.714554776603529), 'dias_positivos': np.int64(26), 'dias_negativos': np.int64(33), 'volatilidad': 1.8235432567719365}


In [40]:
# Prueba de indicadores técnicos
print("\n-- Media Móvil (5 días) --")
ma5 = media_movil(PRECIOS_ACCION, 5)
print(ma5.tail())

print("\n-- Bandas de Bollinger --")
bandas = bandas_bollinger(PRECIOS_ACCION, 20, 2)
for nombre, serie in bandas.items():
    if serie is not None:
        print(f"{nombre}: {serie.iloc[-1]:.2f}")

print("\n-- Tendencia --")
tendencia = clasificar_tendencia(PRECIOS_ACCION)
print(f"Tendencia actual: {tendencia}")


-- Media Móvil (5 días) --
2024-03-18    88.776
2024-03-19    89.318
2024-03-20    89.986
2024-03-21    90.564
2024-03-22    91.134
Freq: B, Name: ACME Corp, dtype: float64

-- Bandas de Bollinger --
banda_superior: 93.65
banda_media: 89.85
banda_inferior: 86.06

-- Tendencia --
Tendencia actual: ALCISTA


In [41]:
# Prueba del reporte completo
print("\nGENERANDO REPORTE COMPLETO...\n")
reporte = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
mostrar_reporte(reporte)


GENERANDO REPORTE COMPLETO...

           REPORTE DE ANÁLISIS: ACME Corp

📅 PERÍODO DE ANÁLISIS
----------------------------------------
Inicio: 2024-01-01
Fin: 2024-03-22
Días analizados: 60

📊 ESTADÍSTICAS DE PRECIO
----------------------------------------
Precio actual:  $92.74
Precio mínimo:  $86.67
Precio máximo:  $111.42
Precio promedio: $96.89

📈 RENDIMIENTO
----------------------------------------
Rendimiento total: -7.75%
Rendimiento promedio diario: -0.131%
Mejor día: 2024-02-13 (+3.91%)
Peor día: 2024-02-21 (-3.71%)
Días positivos: 26
Días negativos: 33

🎯 INDICADORES
----------------------------------------
Tendencia: ALCISTA
Volatilidad: MEDIA
Señal actual: MANTENER



In [42]:
# Comparar las tres acciones
print("\n" + "=" * 70)
print("         COMPARACIÓN DE ACCIONES")
print("=" * 70)

acciones = [
    (PRECIOS_ACCION, "ACME Corp"),
    (ACCION_VOLATIL, "VolatilTech"),
    (ACCION_BAJISTA, "DeclineCorp")
]

for precios, nombre in acciones:
    rendimientos = calcular_rendimientos(precios)
    if rendimientos is not None:
        rend_total = rendimientos.sum() if not rendimientos.isna().all() else 0
        volatilidad = clasificar_volatilidad(rendimientos)
        tendencia = clasificar_tendencia(precios)
        
        print(f"\n{nombre}:")
        print(f"  Rendimiento: {rend_total:+.2f}%")
        print(f"  Volatilidad: {volatilidad}")
        print(f"  Tendencia: {tendencia}")


         COMPARACIÓN DE ACCIONES

ACME Corp:
  Rendimiento: -7.75%
  Volatilidad: MEDIA
  Tendencia: ALCISTA

VolatilTech:
  Rendimiento: +33.35%
  Volatilidad: MUY ALTA
  Tendencia: ALCISTA

DeclineCorp:
  Rendimiento: -33.81%
  Volatilidad: MEDIA
  Tendencia: BAJISTA


## Salida Esperada

```
======================================================================
           REPORTE DE ANÁLISIS: ACME Corp
======================================================================

📅 PERÍODO DE ANÁLISIS
----------------------------------------
Inicio: 2024-01-01
Fin: 2024-03-22
Días analizados: 60

📊 ESTADÍSTICAS DE PRECIO
----------------------------------------
Precio actual:  $112.45
Precio mínimo:  $95.23
Precio máximo:  $118.67
Precio promedio: $106.34

📈 RENDIMIENTO
----------------------------------------
Rendimiento total: +12.45%
Rendimiento promedio diario: +0.207%
Mejor día: 2024-02-15 (+4.32%)
Peor día: 2024-01-23 (-3.87%)
Días positivos: 34
Días negativos: 25

🎯 INDICADORES
----------------------------------------
Tendencia: ALCISTA
Volatilidad: MEDIA
Señal actual: MANTENER

⚠️ ALERTAS RECIENTES
----------------------------------------
🔺 2024-03-05: SUBIDA de +5.23%
🔻 2024-03-12: CAIDA de -5.01%

======================================================================
```

## Bonus: Funcionalidades Extra (Opcional)

### 1. RSI (Relative Strength Index)

In [43]:
def calcular_rsi(precios: pd.Series, periodos: int = 14) -> pd.Series:
    """
    Calcula el RSI (Relative Strength Index).
    """
    cambios = precios.diff()
    
    ganancias = cambios.clip(lower=0)
    perdidas = -1 * cambios.clip(upper=0)
    
    avg_ganancia = ganancias.rolling(window=periodos, min_periods=1).mean()
    avg_perdida = perdidas.rolling(window=periodos, min_periods=1).mean()
    
    rs = avg_ganancia / avg_perdida
    rsi = 100 - (100 / (1 + rs))
    
    rsi[avg_perdida == 0] = 100
    
    return rsi

### 2. Backtesting simple

In [44]:
def backtest_estrategia(precios: pd.Series, senales: pd.Series, capital_inicial: float = 10000) -> Dict:
    """
    Simula la estrategia de trading y calcula rendimiento.
    """
    capital = capital_inicial
    posicion = 0 
    precio_compra = 0
    operaciones_ganadoras = 0
    num_operaciones = 0
    
    for fecha, senal in senales.items():
        precio_actual = precios[fecha]
        
        if senal == "COMPRA" and posicion == 0:
            posicion = capital / precio_actual
            capital = 0
            precio_compra = precio_actual
            num_operaciones += 1
            
        elif senal == "VENTA" and posicion > 0:
            capital = posicion * precio_actual
            posicion = 0
            if precio_actual > precio_compra:
                operaciones_ganadoras += 1
                
    if posicion > 0:
        capital = posicion * precios.iloc[-1]
        
    rendimiento_total = ((capital / capital_inicial) - 1) * 100
    
    return {
        "capital_final": capital,
        "rendimiento_total": rendimiento_total,
        "num_operaciones": num_operaciones,
        "operaciones_ganadoras": operaciones_ganadoras
    }

---

## Criterios de Evaluación

| Criterio | Puntos |
|----------|--------|
| **Parte 1: Análisis Estadístico** | 30 |
| - `estadisticas_basicas()` completo | 10 |
| - `calcular_rendimientos()` correcto | 10 |
| - `analisis_rendimientos()` completo | 10 |
| **Parte 2: Indicadores Técnicos** | 35 |
| - `media_movil()` funciona correctamente | 8 |
| - `bandas_bollinger()` implementado | 10 |
| - `detectar_maximos_minimos()` funciona | 9 |
| - `clasificar_tendencia()` correcto | 8 |
| **Parte 3: Sistema de Alertas** | 35 |
| - `generar_senales_trading()` funciona | 10 |
| - `alertas_precio()` detecta cambios | 8 |
| - `clasificar_volatilidad()` correcto | 7 |
| - `generar_reporte_completo()` integra todo | 10 |
| **Bonus** | +10 |
| **Total** | 100 (+10) |

---

## Entregables

1. Notebook con todas las funciones implementadas
2. Todas las celdas ejecutadas mostrando resultados
3. Análisis de las 3 acciones de prueba
4. Comentarios explicando la lógica de cada función

---

## Consejos

- Usa los métodos de Series: `.pct_change()`, `.rolling()`, `.cumsum()`
- `.idxmax()` y `.idxmin()` retornan el índice del valor máximo/mínimo
- Recuerda manejar NaN que aparecen al calcular rendimientos y medias móviles
- Prueba cada función individualmente antes de integrar